# MRI → CT synthesis — pix2pix + PatchNCE

Training notebook for Kaggle. Before running:

1. **Add the dataset** — *Add Input* → your uploaded `mri-ct-paired-slices`.
2. **Turn on the GPU** — *Settings* → *Accelerator* → **GPU P100** or **T4 x2**.
3. **Turn on internet** if you clone the repo from GitHub (Settings → Internet).

**Sessions are time-limited and will be killed.** That is expected and handled:
every epoch writes a full checkpoint, and re-running this notebook with
`RESUME = 'auto'` picks up exactly where it stopped — same RNG stream, same
optimizer state, same LR schedule. Do **not** lower `n_epochs` to make a run
"fit" in one session: the LR schedule is defined against the total, so that
would change the learning rate of the epochs you already ran.

## 1. Get the code

In [ ]:
import os, sys, subprocess

# Option A — clone from GitHub (needs Internet enabled in Settings).
REPO_URL = 'https://github.com/Ayan2582/MRI_CT_preprocessing_pipeline.git'
REPO_DIR = '/kaggle/working/MRI_CT_preprocessing_pipeline'

# Option B — upload the `model/` folder as a second Kaggle dataset and point
# REPO_DIR at it instead. Nothing outside model/ is needed for training.

if not os.path.isdir(REPO_DIR):
    subprocess.run(['git', 'clone', '--depth', '1', REPO_URL, REPO_DIR], check=True)

sys.path.insert(0, REPO_DIR)
os.chdir(REPO_DIR)
print('repo:', REPO_DIR)

In [ ]:
import torch
print('torch      ', torch.__version__)
print('cuda        ', torch.cuda.is_available())
if torch.cuda.is_available():
    print('device      ', torch.cuda.get_device_name(0))
    print('memory      ', round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), 'GB')

# PyYAML and pandas are preinstalled on Kaggle; nothing else is needed.
import yaml, pandas  # noqa: F401
print('deps ok')

## 2. Point the config at the mounted dataset

The manifest ships with **relative** paths, so only `data.root` changes between
this machine and Kaggle. Everything else in the config is identical, which is
what makes a Kaggle run comparable to a local one.

In [ ]:
import glob

candidates = sorted(glob.glob('/kaggle/input/*/manifest.csv'))
assert candidates, 'No dataset with a manifest.csv found. Add it via *Add Input*.'
DATA_ROOT = os.path.dirname(candidates[0])
print('dataset root:', DATA_ROOT)

import pandas as pd
mf = pd.read_csv(os.path.join(DATA_ROOT, 'manifest.csv'))
print(f'{len(mf)} pairs, {mf.subject_id.nunique()} subjects')
print(mf.body_region.value_counts().to_string())

In [ ]:
# ── Experiment selection ────────────────────────────────────────────────────
# The loss ladder, in the order it is meant to be run. All five share one
# architecture — a U-Net judged by a 70x70 PatchGAN — so they vary the objective
# and nothing else:
#   exp0_l1_only    is the GAN earning its keep at all?
#   exp1_pix2pix    the standard recipe, your baseline  <- START HERE
#   exp2_paper      the target loss at textbook weights
#   exp3_nce_heavy  lean on NCE - an open question, not a favourite
#   exp4_nce_max    where does hallucination start?
#
# Then two that change the ARCHITECTURE instead, to StyleGAN2:
#   exp5_stylegan2_vanilla  StyleGAN2's own loss, 1/0/0 - no L1, no NCE.
#                           The mirror of exp0: that is a regression with no
#                           adversary, this is an adversary with no regression.
#                           Expect poor mae_norm; read it for texture, not
#                           accuracy. A reference point, not a candidate.
#   exp6_stylegan2_fitted   same networks at exp2's weights, sized for 1687
#                           slices. Compare this one against exp2_paper - the
#                           loss is identical, so architecture is the only
#                           variable. This is the candidate.
#
# And one that changes NEITHER the objective's terms nor the architecture, but
# the FRAME the reconstruction loss is taken in:
#   exp7_reggan     a small registration net R predicts the residual
#                   misalignment between the generated CT and the real one, and
#                   L1 is scored after applying it. Compare against exp1: same
#                   networks, same weight, one frame apart.
#
#                   RUN THIS BEFORE exp3. exp3 reweights its objective to hedge
#                   against a residual that base.yaml:55-59 admits is unmeasured;
#                   exp7 measures it and logs it every epoch as R_flow_px, in mm.
#                   If R_flow_px converges near 0 there is nothing to hedge
#                   against and exp3 can be skipped entirely.
#                   Watch R_flow_max: if it grows without bound while G_corr
#                   keeps falling, R is absorbing the generator's mistakes as if
#                   they were misalignment - raise loss.lambda_smooth.
#
# And one that changes the DATA rather than the model:
#   exp8_cyclegan   no pairing at all. The 33 training subjects are split into
#                   two disjoint halves; the MRI comes from one, the CT from the
#                   other, so no patient contributes both modalities. Two
#                   generators, two unconditional discriminators, cycle +
#                   identity loss. Validation stays PAIRED, which is the only
#                   reason it is comparable to anything.
#
#                   It is EXPECTED to lose on mae_norm. Report the gap against
#                   exp1 - that gap is what the QC pairing was worth. Read the
#                   sample panels too: a CycleGAN failure that invents a
#                   plausible CT of the wrong anatomy scores like ordinary blur.
CONFIG = 'model/configs/exp1_pix2pix.yaml'

RESUME = 'auto'   # picks up this run's last.pt if one exists

OVERRIDES = [
    f'data.root={DATA_ROOT}',
    f'data.manifest={DATA_ROOT}/manifest.csv',
    f'data.splits={DATA_ROOT}/splits.json',
    'run.out_dir=/kaggle/working/runs',
    'train.batch_size=8',       # ~11 GB at 256px on a P100; drop to 4 if OOM
    'train.n_epochs=200',       # keep this at the TARGET, not what fits today
    'runtime.num_workers=2',    # Kaggle gives 2 vCPU on GPU instances
    'runtime.amp=true',
]

# exp5/exp6 pick their own batch size, and their R1 gamma is DERIVED from it
# (gamma = 0.0002 * 256^2 / batch). Letting the line above stomp on that would
# silently give the run a penalty weight meant for a different batch size.
# exp8 does the same for its own reason: four resident networks instead of two,
# so it declares batch_size 4 and 8 will not fit a T4 at 256px.
if 'stylegan2' in CONFIG or 'cyclegan' in CONFIG:
    OVERRIDES = [o for o in OVERRIDES if not o.startswith('train.batch_size')]
    print(f'{CONFIG.split("/")[-1]}: batch size left to the config file')

# Change nothing else. Any loss variation is a one-line override, e.g.
#   OVERRIDES += ['loss.lambda_nce=0']       -> plain pix2pix
#   OVERRIDES += ['loss.lambda_gan=0']       -> L1-only regression floor
#   OVERRIDES += ['stabilizers.r1.gamma=5']  -> if G_GAN flatlines on exp5/exp6
#   OVERRIDES += ['loss.lambda_smooth=25']   -> if exp7's R_flow_max runs away
print(CONFIG)
print('\n'.join('  ' + o for o in OVERRIDES))

## 3. Sanity check before spending GPU hours

In [ ]:
import logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s %(levelname)-7s %(message)s',
                    datefmt='%H:%M:%S', force=True)

from model.config import load_config
from model.data.manifest import load_manifest
from model.data.splits import load_split
from model.data.dataset import build_datasets

cfg = load_config(CONFIG, OVERRIDES)
manifest = load_manifest(cfg.data.manifest)
split = load_split(cfg.data.splits)
datasets = build_datasets(cfg, manifest, split)

item = datasets['train'][0]
print('A', tuple(item['A'].shape), 'B', tuple(item['B'].shape),
      'range', (float(item['A'].min()), float(item['A'].max())))
print('config hash', cfg['_hash'])
print({k: len(v.df) for k, v in datasets.items()})

## 4. Train

In [ ]:
from model.training.trainer import Trainer

trainer = Trainer(cfg, datasets)
trainer.maybe_resume(RESUME)
trainer.train()

## 5. Read the curves

`val/mae_norm` on the EMA generator is the number that ranks epochs. The GAN
losses are diagnostics, not quality measures — see `docs/gan_evaluation_guide.md`
for what each shape means.

In [ ]:
import json
import matplotlib.pyplot as plt

# Read train_log.jsonl, not metrics.csv. The JSONL carries every field written
# each epoch; metrics.csv from runs before the logging fix froze its header at
# epoch 0 — during GAN warm-up — so D_acc_*, D_total and G_GAN were dropped.
with open(os.path.join(trainer.run_dir, 'train_log.jsonl')) as fh:
    hist = pd.DataFrame([json.loads(l) for l in fh if l.strip()])

fig, ax = plt.subplots(1, 3, figsize=(16, 4))

ax[0].plot(hist.epoch, hist['val/mae_norm'], label='val mae_norm')
b = hist['val/mae_norm'].idxmin()
ax[0].axvline(hist.epoch[b], ls='--', c='green',
              label=f"best {hist['val/mae_norm'][b]:.5f} @ {int(hist.epoch[b])}")
ax[0].set_title('SELECTION METRIC (lower is better)'); ax[0].legend()

for col in ('train/D_acc_real', 'train/D_acc_fake'):
    if col in hist:
        ax[1].plot(hist.epoch, hist[col], label=col.split('/')[1])
ax[1].axhspan(0.6, 0.85, alpha=0.12, color='green')
ax[1].axhline(0.5, ls=':', c='grey'); ax[1].set_ylim(0, 1.05)
ax[1].set_title('D accuracy - green band is healthy'); ax[1].legend()

# G_PL appears only on the StyleGAN2 runs with path-length regularization; the
# `in hist` guard is what lets one plotting cell serve every config, since each
# run writes only the terms its objective actually has.
for col in ('train/G_GAN', 'train/G_L1', 'train/G_NCE', 'train/G_PL', 'train/D_total'):
    if col in hist:
        ax[2].plot(hist.epoch, hist[col], label=col.split('/')[1])
ax[2].set_yscale('log'); ax[2].set_title('loss terms (log)'); ax[2].legend()

for a in ax:
    a.set_xlabel('epoch'); a.grid(alpha=0.3)
plt.tight_layout(); plt.show()

print('columns available:', [c for c in hist.columns if c.startswith('train/')])

In [ ]:
# The fixed sample panel: MRI | real CT | synth CT | error, same slices every time.
from IPython.display import Image, display

panels = sorted(glob.glob(os.path.join(trainer.sample_dir, '*.png')))
if panels:
    print(os.path.basename(panels[-1]))
    display(Image(panels[-1]))
else:
    print('no panels yet — they are written every logging.sample_every epochs')

## 6. Save the results

Everything under `/kaggle/working` is downloadable from the *Output* tab once
the session ends. Checkpoints are ~650 MB each, so only `best.pt` and `last.pt`
are kept — `last.pt` is the one to restore for the next session's resume.

In [ ]:
for path in sorted(glob.glob(os.path.join(trainer.run_dir, '**', '*'), recursive=True)):
    if os.path.isfile(path):
        print(f'{os.path.getsize(path) / 1e6:9.1f} MB  {os.path.relpath(path, trainer.run_dir)}')

print()
print('To continue in a new session: Save Version, then add this run as a')
print("Notebook Output input, restore last.pt, and re-run with RESUME='auto'.")

### Export

Bundles the logs and sample panels into one small zip, and writes an
inference-only copy of the EMA generator. Full checkpoints are listed but
not copied — they are only needed to resume training.

In [ ]:
# ── Export the important stuff, with download links ───────────────────────────
import os, glob, shutil, torch
from IPython.display import FileLink, display, HTML

RUN = trainer.run_dir if 'trainer' in dir() else sorted(glob.glob('/kaggle/working/runs/*'))[-1]
NAME, OUT = os.path.basename(RUN), '/kaggle/working'

# 1. Results bundle — logs + sample panels + config. Small. Always take this.
stage = os.path.join(OUT, f'{NAME}_results')
shutil.rmtree(stage, ignore_errors=True); os.makedirs(stage)
for f in ('metrics.csv', 'train_log.jsonl', 'train.log', 'config.resolved.yaml'):
    if os.path.isfile(os.path.join(RUN, f)):
        shutil.copy(os.path.join(RUN, f), stage)
if os.path.isdir(os.path.join(RUN, 'samples')):
    shutil.copytree(os.path.join(RUN, 'samples'), os.path.join(stage, 'samples'))
zip_path = shutil.make_archive(stage, 'zip', stage); shutil.rmtree(stage)

# 2. Inference-only weights — the EMA generator alone, ~1/3 of a full checkpoint.
ck  = torch.load(os.path.join(RUN, 'checkpoints', 'best.pt'),
                 map_location='cpu', weights_only=False)
ema = (ck['model'].get('ema') or {}).get('ema')
gen_path = os.path.join(OUT, f'{NAME}_generator.pt')
torch.save({'netG': ema if ema is not None else ck['model']['netG'],
            'weights_source': 'EMA' if ema is not None else 'live',
            'epoch': ck['epoch'], 'best_value': ck['best_value'],
            'selection_metric': ck['selection_metric'],
            'config_hash': ck['config_hash'],
            'loss_plan': ck['model']['loss_plan']}, gen_path)

# 3. Copy the checkpoints alongside so they get links too. Big — skip if you
#    do not intend to resume training in a later session.
INCLUDE_CHECKPOINTS = False
links = [zip_path, gen_path]
if INCLUDE_CHECKPOINTS:
    for tag in ('best', 'last'):
        src = os.path.join(RUN, 'checkpoints', f'{tag}.pt')
        if os.path.isfile(src):
            dst = os.path.join(OUT, f'{NAME}_{tag}.pt')
            if not os.path.exists(dst):
                shutil.copy(src, dst)
            links.append(dst)

# 4. Report + clickable links.
#    FileLink resolves relative to the CWD, and the Kaggle file server is rooted
#    at /kaggle/working — so the links must be built from there, with bare
#    filenames. Hence the chdir.
mb = lambda p: os.path.getsize(p) / 1e6
notes = {zip_path: 'ALWAYS take this — logs + sample panels',
         gen_path: f"inference weights ({ck['model'] and 'EMA' if ema is not None else 'live'})"}

print(f'run  : {RUN}')
print(f"best : {ck['selection_metric']} = {ck['best_value']:.5f} @ epoch {ck['epoch']}")
print()

cwd = os.getcwd()
try:
    os.chdir(OUT)
    for path in links:
        name = os.path.basename(path)
        display(HTML(f'<b>{mb(path):.1f} MB</b> &nbsp; {notes.get(path, "resume only")}'))
        display(FileLink(name))
finally:
    os.chdir(cwd)

if not INCLUDE_CHECKPOINTS:
    print()
    print('Full checkpoints NOT exported (set INCLUDE_CHECKPOINTS = True to link them):')
    for tag in ('best', 'last'):
        p = os.path.join(RUN, 'checkpoints', f'{tag}.pt')
        if os.path.isfile(p):
            print(f'  {mb(p):8.1f} MB  {tag}.pt')

print()
print('If the links do not render (committed/batch runs have no file server),')
print('use the Output panel on the right — the files are in /kaggle/working.')

### Resuming — run this BEFORE the training cell in a follow-up session

Add the previous run as a *Notebook Output* input first, then uncomment.

In [ ]:
# import shutil
# RUN_NAME = 'exp1_pix2pix'
# prev = glob.glob(f'/kaggle/input/*/runs/{RUN_NAME}/checkpoints/last.pt')
# assert prev, 'previous checkpoint not found - check the Notebook Output input'
# dst = f'/kaggle/working/runs/{RUN_NAME}/checkpoints'
# os.makedirs(dst, exist_ok=True)
# shutil.copy(prev[0], os.path.join(dst, 'last.pt'))
# print('restored', prev[0])

## 7. Final evaluation — once, at the end

Run this on the **test** split only after the configuration is frozen. Every
time you read a test number and then change something in response, the test set
becomes a second validation set. Six subjects will not survive that repeatedly.

In [ ]:
# trainer.load_checkpoint(os.path.join(trainer.ckpt_dir, 'best.pt'))
# results = trainer.validate(trainer.start_epoch - 1, split='test')
# print(trainer.metrics.format_table(results))